In [1]:
%run 00_config.py

In [ ]:

from encoder_visual_bge import Encoder

model_name = "BAAI/bge-base-en-v1.5"
model_path = "models/Visualized_base_en_v1.5.pth"  # Change to your own value if using a different model path
encoder = Encoder(model_name, model_path)

In [3]:
from pymilvus import MilvusClient

In [4]:
client = MilvusClient(
    uri="http://localhost:19530",
    token="root:Milvus"
)

In [ ]:
if (run_in_cloud):
    client = MilvusClient(
        uri="https://in01-2082e32a6e894f2.gcp-us-central1.vectordb.zillizcloud.com:443", #replace this with your own zilliz cloud cluster
        token="redacted_api_token"
    )

In [114]:
client.flush(
    collection_name=collection_name,
    database_name="default"
)

In [7]:
client.load_collection(
    collection_name=collection_name,
    database_name="default"
)

In [ ]:
client.get_collection_stats(
    collection_name=collection_name
)

In [6]:
results = client.query(
    collection_name=collection_name, 
    filter="product_id == '66d49bbed043f5be260fa9f7fbff5957'",
    output_fields=['product_id', 'image_file', 'prod_imagenum'],
    database_name="default"
)

In [ ]:
for result in results:
    for key, value in result.items():
        print(f"{key}: {value}")

In [34]:
query_list = []
query_list.append( ("fc26bf9c4f82f38d46b9dccc46718d99_71feE_2BPLu0L_7.jpg", "board games similar to this theme") )
#query_list.append( ("0a2c21e52bc3268c3c599302bb18bac9_514N60QheQL_7.jpg", "men's clothing that resembles this") )
# query_list.append( ("37575f213b120dbf2efd50ba654f7ee5_51ynl0BkZAL_7.jpg", "boys playing like this") )
# query_list.append( ("10e48ea4fb85adbf89b0c1393ad25659_41Br%2BZMq1kL_7.jpg", "greeting cards in different colors like this") )
# query_list.append( ("10e48ea4fb85adbf89b0c1393ad25659_41Br%2BZMq1kL_7.jpg", "office supplies similar to this") )
# query_list.append( ("fcfe676348b9b49e27128329cbc9cfa3_510KX_2BBcXGL_7.jpg", "many children playing with stuffed toys") )
#query_list.append( ("f4e5c1c366de996a239b2b6039c8ea7f_41l2Bw6ua5ML_7.jpg", "in blue color") )
query_list.append( ("2f92ca5061ebc2d3e998008408aa6827_51GUZO82OwL_6.jpg", "larger version for adults in darker colors"))
#query_list.append( ("15d9dc31949b787083d1a997fe02e818_514EUPqB18L_6.jpg", "something like this in the color red"))
#query_list.append(("ffafd50e7d4ad0375938f044914bd2f5_51EOzSU1bQL_6.jpg", "similar product in black color"))
query_list.append(("00b62745d1687193dbcd10b0d1c27225_51afXq0m3NL_7.jpg", "something like this for a motorcycle"))

#query_list.append(("00f50ad3ea80832e1999766ee2620ebd_51SrCYwE3LL_7.jpg", "child playing in in the dark"))
#query_list.append(("ffee1075c330d56ec13c6e0fe75435bd_51HaL3Dt1HL_6.jpg", "show me this with cats or rabbit"))

In [ ]:

import IPython.display as IP

IP.Image(filename="product_images/" + query_list[-1][0], width=220)


In [ ]:
result = client.query(
    collection_name=collection_name,
    filter="RANDOM_SAMPLE(0.1)",
    output_fields=["image_file", "prod_text"],
    limit=5,
)
for i in result:
    print(i['image_file'])
    print(i['id'])

In [ ]:
import os
query_dir = dl_images_folder
query_item = query_list [-1]
query_image = os.path.join(
    query_dir, query_item[0]  # Use the query image from the list
)  # Change to your own query image path
query_text = query_item[1] 

# Generate query embedding given image and text instructions
query_vec = encoder.encode_query(image_path=query_image, text=query_text)

search_results = client.search(
    collection_name=collection_name,
    #anns_field=["vector"],
    data=[query_vec],
    output_fields=["id", "product_id","image_file", "prod_imagenum"],
    #output_fields=["*"],
    limit=15,  # Max number of search results to return,
    # group_by_field = "product_id",
    # group_size = 2,
    # search_params={
    #     "metric_type": "COSINE",
    #     "params": {# "level" : 4, 
    #                #"enable_recall_calculation": True
    #                } },  # Search parameters
)
#print(search_results.recalls) # this evaluates to none if not enabled in search_params
print(search_results[0])
import json
json.loads(search_results)
v = search_results[0]
# d1 = dict(v)
print(len(v)  )
for i in v:
    dd = dict(i)
    #print(dd["entity"].keys())
    retrieved_images = [ (hit.get("distance"), hit.get("entity").get("image_file"), hit.get("entity").get("product_id")) for hit in search_results[0]]
images_list = [hit.get("image_file").replace('%','_') for hit in v]


for i, ri in enumerate(retrieved_images):
    print(f"Result {i+1}: Distance: {ri[0]}, Image File: {ri[1]}")
    #print(f"Result {i+1}: {ri}")
# images_list = [os.path.join(f"{query_dir}/", img[1]) for img in retrieved_images]


In [ ]:
print(search_results)

In [ ]:
import os
query_dir = dl_images_folder
query_item = query_list [-1]
query_image = os.path.join(
    query_dir, query_item[0]  # Use the query image from the list
)  # Change to your own query image path
query_text = query_item[1] 

# Generate query embedding given image and text instructions
query_vec = encoder.encode_query(image_path=query_image, text=query_text)

search_results = client.search(
    collection_name=collection_name,
    #anns_field="vector",
    data=[query_vec],
    output_fields=["id", "product_id","image_file", "prod_imagenum"],
    #output_fields=["*"],
    limit=12,  # Max number of search results to return,
    # group_by_field = "product_id",
    # group_size = 2,
    search_params={
        "metric_type": "COSINE",
        "params": { "level" : 4, 
                   #"enable_recall_calculation": True
                   } },  # Search parameters
)
print(search_results[0])
v = search_results[0]
# d1 = dict(v)
print(len(v)  )
for i in v:
    dd = dict(i)
    #print(dd["entity"].keys())
    retrieved_images = [ (hit.get("distance"), hit.get("entity").get("image_file"), hit.get("entity").get("product_id")) for hit in search_results[0]]
images_list = [hit.get("image_file").replace('%','_') for hit in v]


for i, ri in enumerate(retrieved_images):
    print(f"Result {i+1}: {ri}")
# images_list = [os.path.join(f"{query_dir}/", img[1]) for img in retrieved_images]


In [ ]:
print(images_list)

In [ ]:
import ipyplot

ipyplot.plot_images(images_list, max_images=15, img_width=200)
